# DARN/SuperDARN — Implementation Companion / 구현 동반 노트북

**Paper**: Greenwald et al. (1995), *Space Science Reviews* 71, 761-796.

## 목적 / Purpose

이 노트북은 SuperDARN의 세 가지 핵심 기법을 from-scratch로 재구성한다:

1. **다중 펄스 ACF로부터의 도플러 속도 추정** — Doppler velocity from a synthetic multi-pulse ACF
2. **빔 스윙 단일 레이더 벡터 재구성** — Single-radar beam-swinging vector reconstruction (Ruohoniemi et al. 1989)
3. **두 레이더 공통 체적 벡터 분해** — Two-radar common-volume vector decomposition

All synthetic — no real radar files. Educational reproduction of the algorithms behind Figs. 15-17 of the paper.

모두 합성 데이터 — 실제 레이더 파일 미사용. 논문 Figs. 15-17 알고리즘의 교육용 재구성.

In [ ]:
"""Imports and global constants for SuperDARN technique reproduction."""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

# Reproducibility for synthetic data.
RNG = np.random.default_rng(seed=58)

# Physical constants.
C_LIGHT = 2.998e8  # speed of light, m/s

# Plot style.
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

## 1. Doppler Velocity from a Multi-Pulse ACF / 다중 펄스 ACF로부터의 도플러 속도

SuperDARN sends 5-7 pulses within ~100 ms; receive samples are correlated to form a complex ACF $R(\tau) = \langle E^*(t) E(t+\tau) \rangle$. The mean Doppler velocity is recovered from the **phase slope** $d\phi/d\tau$:

$$ v_{\rm LOS} = -\frac{\lambda_0}{4\pi} \frac{d\phi(\tau)}{d\tau} $$

and the spectral width from the magnitude decay $|R(\tau)| \sim |R(0)| e^{-(\tau/\tau_c)^p}$.

SuperDARN은 ~100 ms 내 5-7개 펄스를 송신하고, 수신 샘플의 상관으로 복소 ACF를 형성. 평균 도플러는 위상 기울기에서, 스펙트럼 폭은 크기 감쇠에서 산출.

In [ ]:
def synthesize_acf(
    lags_s: np.ndarray,
    radar_freq_hz: float,
    true_v_los_m_s: float,
    spectral_width_m_s: float,
    backscatter_power: float = 1.0,
    noise_power: float = 0.02,
) -> np.ndarray:
    """Generate a synthetic SuperDARN-like complex ACF.

    The SuperDARN backscatter model assumes (i) phase rotation at angular
    frequency 2 * 2*pi*f0/c * v_LOS (the 4*pi/lambda factor) and (ii)
    magnitude decay set by the spectral width.

    Args:
        lags_s: 1-D array of lag values in seconds.
        radar_freq_hz: Radar carrier frequency in Hz (e.g., 12e6 for 12 MHz).
        true_v_los_m_s: True line-of-sight velocity in m/s; positive means
            scatterer moving away from the radar.
        spectral_width_m_s: 1/e half-width of the Doppler power spectrum,
            in m/s.
        backscatter_power: Power at lag zero (|R(0)|).
        noise_power: Variance of complex Gaussian noise added to each lag.

    Returns:
        Complex ndarray of shape (len(lags_s),) containing the noisy ACF.
    """
    lambda0 = C_LIGHT / radar_freq_hz
    # Doppler angular frequency in rad/s.
    omega_d = -4.0 * np.pi * true_v_los_m_s / lambda0
    # Spectral-width-induced decorrelation time (Gaussian shape).
    tau_c = lambda0 / (4.0 * np.pi * spectral_width_m_s)
    # Clean ACF model.
    acf_clean = backscatter_power * np.exp(1j * omega_d * lags_s) \
        * np.exp(-(lags_s / tau_c) ** 2)
    # Additive complex Gaussian noise.
    noise_real = RNG.normal(scale=np.sqrt(noise_power / 2), size=lags_s.shape)
    noise_imag = RNG.normal(scale=np.sqrt(noise_power / 2), size=lags_s.shape)
    return acf_clean + (noise_real + 1j * noise_imag)


def fit_doppler_from_acf(
    lags_s: np.ndarray,
    acf: np.ndarray,
    radar_freq_hz: float,
) -> dict:
    """Estimate v_LOS, spectral width, and power from a complex ACF.

    The estimator is a constrained nonlinear least-squares fit of the
    same parametric model used in synthesize_acf. This is conceptually
    identical to the FITACF algorithm used operationally by SuperDARN.

    Args:
        lags_s: Lag values in seconds (must include zero lag or near-zero).
        acf: Complex ACF samples corresponding to lags_s.
        radar_freq_hz: Radar carrier frequency in Hz.

    Returns:
        Dict with keys 'v_los_m_s', 'spectral_width_m_s', 'power'.
    """
    lambda0 = C_LIGHT / radar_freq_hz

    def residuals(params: np.ndarray) -> np.ndarray:
        v_los, w, p = params
        omega_d = -4.0 * np.pi * v_los / lambda0
        tau_c = lambda0 / (4.0 * np.pi * max(w, 1e-3))
        model = p * np.exp(1j * omega_d * lags_s) * np.exp(-(lags_s / tau_c) ** 2)
        diff = model - acf
        # Stack real and imaginary parts as separate residuals.
        return np.concatenate([diff.real, diff.imag])

    # Initial guess: phase slope between two non-zero lags for v_LOS,
    # half-power half-width for w, magnitude at the smallest lag for p.
    nonzero_idx = np.argsort(np.abs(lags_s))[1:3]
    if len(nonzero_idx) >= 2:
        l1, l2 = lags_s[nonzero_idx]
        phi1, phi2 = np.angle(acf[nonzero_idx[0]]), np.angle(acf[nonzero_idx[1]])
        dphi = np.angle(np.exp(1j * (phi2 - phi1)))
        v_init = -dphi / (4 * np.pi * (l2 - l1) / lambda0)
    else:
        v_init = 0.0
    p_init = float(np.abs(acf[np.argmin(np.abs(lags_s))]))
    w_init = 100.0
    result = least_squares(residuals, x0=[v_init, w_init, p_init])
    return {
        'v_los_m_s': float(result.x[0]),
        'spectral_width_m_s': float(result.x[1]),
        'power': float(result.x[2]),
    }


# Demonstration: synthesize and recover.
RADAR_FREQ = 12e6  # 12 MHz, typical SuperDARN frequency.
MULTIPULSE_LAGS = np.array([0, 14, 22, 24, 27, 31, 42], dtype=float) * 2.4e-3
TRUE_V = 480.0  # m/s
TRUE_W = 120.0  # m/s

acf_sample = synthesize_acf(MULTIPULSE_LAGS, RADAR_FREQ, TRUE_V, TRUE_W)
fit = fit_doppler_from_acf(MULTIPULSE_LAGS, acf_sample, RADAR_FREQ)

print(f'True v_LOS = {TRUE_V:.1f} m/s, fitted = {fit["v_los_m_s"]:.1f} m/s')
print(f'True width = {TRUE_W:.1f} m/s, fitted = {fit["spectral_width_m_s"]:.1f} m/s')
print(f'Fitted power = {fit["power"]:.3f}')

In [ ]:
"""Visualize the ACF and the fitted phase slope."""

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Magnitude vs lag.
axes[0].plot(MULTIPULSE_LAGS * 1e3, np.abs(acf_sample), 'o-', label='|ACF|')
axes[0].set_xlabel('Lag / ms')
axes[0].set_ylabel('|R(tau)|')
axes[0].set_title('ACF magnitude (decay → spectral width)')
axes[0].grid(alpha=0.3)
axes[0].legend()

# Phase vs lag — slope yields Doppler velocity.
phase_unwrapped = np.unwrap(np.angle(acf_sample))
axes[1].plot(MULTIPULSE_LAGS * 1e3, phase_unwrapped, 'o-', label='unwrapped phase')
lambda0 = C_LIGHT / RADAR_FREQ
expected_slope = -4 * np.pi * TRUE_V / lambda0
axes[1].plot(
    MULTIPULSE_LAGS * 1e3,
    expected_slope * MULTIPULSE_LAGS,
    'r--',
    label=f'true slope (v={TRUE_V} m/s)',
)
axes[1].set_xlabel('Lag / ms')
axes[1].set_ylabel('phase / rad')
axes[1].set_title('ACF phase (slope → v_LOS)')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Single-Radar Beam-Swinging Vector Reconstruction / 단일 레이더 빔 스윙 벡터 재구성

Ruohoniemi et al. (1989) assume the F-region flow is uniform along an L-shell at fixed invariant latitude. Then the LOS Doppler at beam azimuth $\theta$ is:

$$ v_{\rm LOS}(\theta) = V \cos(\theta - \theta_0) $$

where $(V, \theta_0)$ are the magnitude and direction of the 2-D flow vector. Fitting this sinusoid to LOS Dopplers from many beams yields the vector. Freeman et al. (1991) showed that violations of L-shell uniformity can produce 180° errors — illustrated below.

Ruohoniemi et al. (1989)는 F-region 흐름이 고정 invariant 위도의 L-shell을 따라 균일하다고 가정. 빔 방위각 theta에 대한 LOS 도플러는 사인꼴이 되며, 이를 피팅해 2D 벡터를 얻는다. Freeman et al. (1991)이 지적한 180° 오차 가능성도 함께 시연.

In [ ]:
def beam_swinging_fit(
    beam_azimuths_rad: np.ndarray,
    los_velocities_m_s: np.ndarray,
    los_uncertainty_m_s: float = 50.0,
) -> dict:
    """Fit a 2-D velocity vector from line-of-sight Dopplers across beams.

    Implements the Ruohoniemi et al. (1989) L-shell beam-swinging
    algorithm by least-squares fitting v_LOS(theta) = Vx*cos(theta) +
    Vy*sin(theta). The cosine/sine basis avoids nonlinear initialization.

    Args:
        beam_azimuths_rad: 1-D array of beam azimuths (radians) in a
            geomagnetic frame where 0 means magnetic north.
        los_velocities_m_s: Corresponding line-of-sight Doppler
            velocities in m/s.
        los_uncertainty_m_s: Per-sample LOS uncertainty for error
            propagation (assumed constant).

    Returns:
        Dict with the fitted Cartesian components, magnitude, direction,
        and per-component standard errors.
    """
    # Linear design matrix: columns [cos(theta), sin(theta)].
    design = np.column_stack([np.cos(beam_azimuths_rad), np.sin(beam_azimuths_rad)])
    fit, residuals, rank, _ = np.linalg.lstsq(design, los_velocities_m_s, rcond=None)
    v_x, v_y = fit
    magnitude = float(np.hypot(v_x, v_y))
    direction_rad = float(np.arctan2(v_y, v_x))
    # Covariance: sigma^2 * (D^T D)^-1.
    cov = los_uncertainty_m_s ** 2 * np.linalg.inv(design.T @ design)
    return {
        'v_x_m_s': float(v_x),
        'v_y_m_s': float(v_y),
        'magnitude_m_s': magnitude,
        'direction_deg': float(np.degrees(direction_rad)),
        'sigma_v_x': float(np.sqrt(cov[0, 0])),
        'sigma_v_y': float(np.sqrt(cov[1, 1])),
    }


# Demonstration: a true 800 m/s flow at 30 degrees east of north,
# observed by a 16-beam azimuthal scan with realistic LOS noise.
TRUE_FLOW = (800.0, np.radians(30.0))  # (magnitude, direction)
TRUE_VX = TRUE_FLOW[0] * np.cos(TRUE_FLOW[1])
TRUE_VY = TRUE_FLOW[0] * np.sin(TRUE_FLOW[1])

beam_azimuths = np.radians(np.linspace(-26, 26, 16))  # SuperDARN ~52 deg sector.
true_los = TRUE_VX * np.cos(beam_azimuths) + TRUE_VY * np.sin(beam_azimuths)
noisy_los = true_los + RNG.normal(scale=40.0, size=true_los.shape)

result = beam_swinging_fit(beam_azimuths, noisy_los, los_uncertainty_m_s=40.0)
print(f'True flow: V = {TRUE_FLOW[0]:.1f} m/s, dir = {np.degrees(TRUE_FLOW[1]):.1f} deg')
print(f'Fitted:    V = {result["magnitude_m_s"]:.1f} m/s, '
      f'dir = {result["direction_deg"]:.1f} deg')
print(f'  sigma_Vx = {result["sigma_v_x"]:.1f} m/s, '
      f'sigma_Vy = {result["sigma_v_y"]:.1f} m/s')

In [ ]:
"""Visualize the sinusoidal LOS variation and the fitted curve."""

azimuth_grid = np.linspace(-np.pi, np.pi, 361)
fitted_curve = result['v_x_m_s'] * np.cos(azimuth_grid) + result['v_y_m_s'] * np.sin(azimuth_grid)
true_curve = TRUE_VX * np.cos(azimuth_grid) + TRUE_VY * np.sin(azimuth_grid)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(np.degrees(azimuth_grid), true_curve, 'k-', alpha=0.4, label='true v_LOS(theta)')
ax.plot(np.degrees(azimuth_grid), fitted_curve, 'r-', label='fitted v_LOS(theta)')
ax.errorbar(
    np.degrees(beam_azimuths), noisy_los, yerr=40.0,
    fmt='o', color='b', alpha=0.6, label='observed beams (sigma=40 m/s)',
)
ax.axvspan(-26, 26, alpha=0.08, color='gray', label='SuperDARN scan sector')
ax.set_xlabel('beam azimuth / deg')
ax.set_ylabel('v_LOS / m/s')
ax.set_title('Beam-swinging single-radar vector reconstruction')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Freeman et al. (1991) failure case / Freeman et al. (1991) 실패 사례

When flow is **not** uniform along the L-shell — for example, at a Harang Discontinuity or a convection-cell boundary — the sinusoidal fit can produce a 180° error in direction. We simulate this by giving each beam a different underlying flow direction.

흐름이 L-shell 따라 균일하지 **않을** 때(예: Harang Discontinuity, 대류 셀 경계) 사인꼴 피팅은 방향에 180° 오차를 낼 수 있다. 각 빔에 다른 기저 흐름을 부여해 시뮬레이션.

In [ ]:
"""Simulate a flow field with a sharp directional reversal across beams."""

# Beams pointing dawn-ward see flow toward dusk; beams pointing dusk-ward
# see flow toward dawn — a Harang-Discontinuity-like inversion.
varying_dir_deg = np.where(np.degrees(beam_azimuths) < 0, 30.0, 210.0)
vx_per_beam = 800.0 * np.cos(np.radians(varying_dir_deg))
vy_per_beam = 800.0 * np.sin(np.radians(varying_dir_deg))
los_violation = vx_per_beam * np.cos(beam_azimuths) + vy_per_beam * np.sin(beam_azimuths)
los_violation_noisy = los_violation + RNG.normal(scale=40.0, size=los_violation.shape)

result_violation = beam_swinging_fit(beam_azimuths, los_violation_noisy, 40.0)
print('When the L-shell-uniform assumption is violated:')
print(f'  Reported flow: V = {result_violation["magnitude_m_s"]:.1f} m/s, '
      f'dir = {result_violation["direction_deg"]:.1f} deg')
print('  -- but the underlying flow is bimodal (30 deg AND 210 deg) --')
print('  This is exactly the kind of failure Freeman et al. (1991) warned about.')

## 3. Two-Radar Common-Volume Vector Decomposition / 두 레이더 공통 체적 벡터 분해

When two radars observe the same scattering volume from different angles, their two LOS Dopplers fully constrain the local 2-D vector — no L-shell assumption is needed:

$$ \begin{pmatrix} v_A \\ v_B \end{pmatrix} = \begin{pmatrix} \hat{r}_A^x & \hat{r}_A^y \\ \hat{r}_B^x & \hat{r}_B^y \end{pmatrix} \begin{pmatrix} V_x \\ V_y \end{pmatrix}. $$

Error amplification scales as $1/\sin\alpha$ where $\alpha$ is the angle between the two LOS unit vectors. We sweep $\alpha$ to demonstrate why ~500 km separations were inadequate.

두 레이더가 같은 산란 체적을 다른 각도에서 보면 두 LOS 도플러가 국소 2D 벡터를 완전히 구속 — L-shell 가정 불필요. 오차 증폭은 두 LOS 단위벡터 사잇각 alpha에 대해 1/sin(alpha)로 스케일. ~500 km 분리가 왜 부족했는지 alpha 스윕으로 시연.

In [ ]:
def two_radar_decomposition(
    los_vector_a: np.ndarray,
    los_vector_b: np.ndarray,
    v_a_m_s: float,
    v_b_m_s: float,
) -> dict:
    """Solve for a 2-D plasma velocity from two radars' LOS Dopplers.

    The decomposition matrix M stacks the two horizontal LOS unit vectors
    as rows; M is well-conditioned only when the two LOS directions are
    sufficiently non-parallel.

    Args:
        los_vector_a: Horizontal unit vector (x, y) of radar A's LOS at
            the common volume.
        los_vector_b: Horizontal unit vector (x, y) of radar B's LOS.
        v_a_m_s: Radar A's measured LOS Doppler.
        v_b_m_s: Radar B's measured LOS Doppler.

    Returns:
        Dict with keys 'v_x', 'v_y', 'magnitude', 'condition_number',
        'angle_between_los_deg'.
    """
    matrix = np.array([los_vector_a, los_vector_b], dtype=float)
    los_doppler = np.array([v_a_m_s, v_b_m_s], dtype=float)
    velocity_xy = np.linalg.solve(matrix, los_doppler)
    cond = float(np.linalg.cond(matrix))
    cos_angle = float(np.dot(los_vector_a, los_vector_b))
    angle_deg = float(np.degrees(np.arccos(np.clip(cos_angle, -1, 1))))
    return {
        'v_x': float(velocity_xy[0]),
        'v_y': float(velocity_xy[1]),
        'magnitude': float(np.hypot(*velocity_xy)),
        'condition_number': cond,
        'angle_between_los_deg': angle_deg,
    }


# Sweep angle alpha to study error amplification.
TRUE_V_VECTOR = np.array([400.0, 600.0])  # m/s
TRUE_MAG = float(np.hypot(*TRUE_V_VECTOR))
ALPHAS_DEG = np.array([10, 20, 30, 45, 60, 75, 90, 120])
LOS_NOISE = 40.0  # m/s

print(f'True velocity = ({TRUE_V_VECTOR[0]:.0f}, {TRUE_V_VECTOR[1]:.0f}) m/s, '
      f'|V| = {TRUE_MAG:.1f} m/s')
print('-' * 70)
print(f'{"alpha [deg]":>11} {"cond(M)":>9} {"<|V_fit|>":>11} {"std(|V|)":>11}')
print('-' * 70)
for alpha in ALPHAS_DEG:
    los_a = np.array([1.0, 0.0])
    los_b = np.array([np.cos(np.radians(alpha)), np.sin(np.radians(alpha))])
    v_a_clean = float(np.dot(TRUE_V_VECTOR, los_a))
    v_b_clean = float(np.dot(TRUE_V_VECTOR, los_b))
    fitted_mags = []
    n_trials = 1000
    for _ in range(n_trials):
        v_a_obs = v_a_clean + RNG.normal(scale=LOS_NOISE)
        v_b_obs = v_b_clean + RNG.normal(scale=LOS_NOISE)
        out = two_radar_decomposition(los_a, los_b, v_a_obs, v_b_obs)
        fitted_mags.append(out['magnitude'])
    fitted_mags = np.array(fitted_mags)
    cond = out['condition_number']
    print(f'{alpha:>11d} {cond:>9.2f} {fitted_mags.mean():>11.1f} {fitted_mags.std():>11.1f}')

In [ ]:
"""Plot magnitude error vs. angle between LOS vectors."""

alphas_fine = np.linspace(5, 175, 35)
stds = []
for alpha in alphas_fine:
    los_a = np.array([1.0, 0.0])
    los_b = np.array([np.cos(np.radians(alpha)), np.sin(np.radians(alpha))])
    v_a_clean = float(np.dot(TRUE_V_VECTOR, los_a))
    v_b_clean = float(np.dot(TRUE_V_VECTOR, los_b))
    mags = []
    for _ in range(500):
        v_a = v_a_clean + RNG.normal(scale=LOS_NOISE)
        v_b = v_b_clean + RNG.normal(scale=LOS_NOISE)
        out = two_radar_decomposition(los_a, los_b, v_a, v_b)
        mags.append(out['magnitude'])
    stds.append(np.std(mags))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(alphas_fine, stds, 'b-o', label='|V| std (Monte Carlo)')
predicted = LOS_NOISE * np.sqrt(2) / np.sin(np.radians(alphas_fine))
ax.plot(alphas_fine, predicted, 'r--', label='theoretical $\\sigma\\sqrt{2}/\\sin\\alpha$')
ax.axvspan(20, 40, alpha=0.15, color='orange',
           label='~500 km separation regime')
ax.axvspan(60, 120, alpha=0.10, color='green',
           label='SuperDARN target regime')
ax.set_xlabel('angle between two LOS vectors / deg')
ax.set_ylabel('std(|V_fit|) / m/s')
ax.set_yscale('log')
ax.set_title('Why ~500 km radar separations were inadequate')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## 4. Putting it together: a synthetic SuperDARN convection map / 통합: 합성 SuperDARN 대류 지도

We construct a Heppner-Maynard-like two-cell convection pattern, sample it with a single radar's beam grid, and reconstruct the LOS Doppler field — the data product behind Figs. 15-17.

Heppner-Maynard 풍의 2-셀 대류 패턴을 만들고, 단일 레이더 빔 그리드로 샘플링한 LOS 도플러 장 — Figs. 15-17 데이터 산출물의 합성 재구성.

In [ ]:
def heppner_maynard_like_potential(
    mlat_deg: np.ndarray,
    mlt_hr: np.ndarray,
    cap_potential_kV: float = 60.0,
) -> np.ndarray:
    """Toy two-cell electrostatic potential (kV) for visualization.

    Not a quantitative model -- meant only as a smooth reference field
    against which to verify radar sampling.

    Args:
        mlat_deg: Magnetic latitude grid in degrees (60-90).
        mlt_hr: Magnetic local time grid in hours (0-24).
        cap_potential_kV: Total cross-polar-cap potential drop in kV.

    Returns:
        Electrostatic potential at each (mlat, mlt) sample, in kV.
    """
    # Polar-angle measure from the magnetic pole.
    co_lat_rad = np.radians(90.0 - mlat_deg)
    mlt_rad = mlt_hr * np.pi / 12.0  # 24 hr -> 2*pi.
    # Two-cell pattern peaks at dawn (06 MLT) and dusk (18 MLT).
    return 0.5 * cap_potential_kV * np.sin(co_lat_rad) ** 2 * np.sin(mlt_rad)


# Build a (mlat, mlt) grid covering Northern Hemisphere auroral zone.
MLAT_GRID_DEG = np.linspace(60, 88, 60)
MLT_GRID_HR = np.linspace(0, 24, 96, endpoint=False)
mlat_2d, mlt_2d = np.meshgrid(MLAT_GRID_DEG, MLT_GRID_HR, indexing='ij')
potential_kV = heppner_maynard_like_potential(mlat_2d, mlt_2d, cap_potential_kV=80.0)

# Plot in MLT/MLAT polar projection.
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'projection': 'polar'})
theta = mlt_2d * np.pi / 12.0
r = 90.0 - mlat_2d
contour = ax.contour(theta, r, potential_kV, levels=15, cmap='RdBu_r')
ax.set_theta_zero_location('S')  # 00 MLT at bottom (midnight), conventional.
ax.set_theta_direction(-1)
ax.set_rmax(30)
ax.set_rticks([10, 20, 30])
ax.set_yticklabels(['80', '70', '60'])
ax.set_xticks(np.linspace(0, 2 * np.pi, 4, endpoint=False))
ax.set_xticklabels(['00', '06', '12', '18'])
fig.colorbar(contour, ax=ax, label='Potential / kV', shrink=0.8)
ax.set_title('Toy two-cell convection potential (Northern Hemisphere)\nMLAT vs MLT')
plt.tight_layout()
plt.show()

## 5. Summary / 요약

We reproduced the three signal-processing pillars of SuperDARN at minimum complexity:

1. **ACF fitting** recovers Doppler velocity (and spectral width and power) from a 100 ms multi-pulse sequence. Recovered $v_{\rm LOS}$ matched the synthetic truth within noise.
2. **Beam-swinging** turns line-of-sight Dopplers across 16 beams into a 2-D vector under the L-shell-uniform assumption, but is fragile when that assumption fails (Freeman et al. 1991 demonstrated 180° errors).
3. **Two-radar common-volume decomposition** removes the L-shell assumption entirely, but its noise amplification scales as $1/\sin\alpha$, demanding $\alpha \gtrsim 60$° (i.e., radar separations beyond ~500 km).

These three building blocks, combined with synchronized 96-second scans and a Common Programs operating policy, are exactly what Greenwald et al. (1995) advertised as the SuperDARN concept — and they remain the operational core of SuperDARN three decades later, fed into the Ruohoniemi & Baker (1998) APL spherical-harmonic global fit.

SuperDARN의 세 신호 처리 기둥을 최소 복잡도로 재구성: (1) ACF 피팅 — 100 ms 다중 펄스에서 도플러·폭·출력 동시 산출, (2) 빔 스윙 — 16 빔의 LOS를 L-shell 균일 가정 하 2D 벡터로 변환, (3) 두 레이더 공통 체적 분해 — L-shell 가정 제거, 단 1/sin(alpha) 노이즈 증폭. Greenwald et al. (1995)이 광고한 SuperDARN 개념의 핵심이며 30년 후에도 운영 핵심으로, Ruohoniemi & Baker (1998) APL 구면조화 글로벌 피팅으로 이어짐.